# Project FORESIGHT — Data Cleaning

**Goal:** Load the four provided Excel extracts, inspect data quality, clean dates/columns/duplicates, validate relationships, merge the datasets, and save an analysis-ready dataset.

Datasets:
- `sales_daily`
- `sku_master`
- `inventory_snapshots`
- `calendar`


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)


## 1. File paths

In [ ]:
# If this notebook is saved inside the project root, keep the paths below.
# If your Excel files are in another folder, update DATA_DIR.

DATA_DIR = "../data"

sales_path = os.path.join(DATA_DIR, "sales_daily (1).xlsx")
sku_path = os.path.join(DATA_DIR, "sku_master (1).xlsx")
inventory_path = os.path.join(DATA_DIR, "inventory_snapshots (1).xlsx")
calendar_path = os.path.join(DATA_DIR, "calendar (1).xlsx")

print("Data directory:", DATA_DIR)


## 2. Load the four Excel files

In [ ]:
sales = pd.read_excel(sales_path)
sku = pd.read_excel(sku_path)
inventory = pd.read_excel(inventory_path)
calendar = pd.read_excel(calendar_path)

print("Files loaded successfully.")


In [ ]:
print("Sales shape      :", sales.shape)
print("SKU master shape :", sku.shape)
print("Inventory shape  :", inventory.shape)
print("Calendar shape   :", calendar.shape)


## 3. Standardize column names

In [ ]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df

sales = clean_column_names(sales)
sku = clean_column_names(sku)
inventory = clean_column_names(inventory)
calendar = clean_column_names(calendar)

print("Sales columns:", sales.columns.tolist())
print("SKU columns:", sku.columns.tolist())
print("Inventory columns:", inventory.columns.tolist())
print("Calendar columns:", calendar.columns.tolist())


## 4. Initial inspection

In [ ]:
print("SALES")
display(sales.head())

print("\nSKU MASTER")
display(sku.head())

print("\nINVENTORY")
display(inventory.head())

print("\nCALENDAR")
display(calendar.head())


In [ ]:
print("Data types — Sales")
display(sales.dtypes.to_frame("dtype"))

print("Data types — SKU Master")
display(sku.dtypes.to_frame("dtype"))

print("Data types — Inventory")
display(inventory.dtypes.to_frame("dtype"))

print("Data types — Calendar")
display(calendar.dtypes.to_frame("dtype"))


## 5. Convert date columns

In [ ]:
sales["date"] = pd.to_datetime(sales["date"], errors="coerce")
calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")
inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"], errors="coerce"
)

print("Sales date range:", sales["date"].min(), "to", sales["date"].max())
print("Calendar date range:", calendar["date"].min(), "to", calendar["date"].max())
print("Inventory restock date range:",
      inventory["last_restock_date"].min(), "to",
      inventory["last_restock_date"].max())


## 6. Missing-value report

In [ ]:
def missing_report(df):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })
    return report.sort_values("missing_percent", ascending=False)

print("Sales")
display(missing_report(sales))

print("SKU Master")
display(missing_report(sku))

print("Inventory")
display(missing_report(inventory))

print("Calendar")
display(missing_report(calendar))


## 7. Duplicate checks

In [ ]:
print("Exact duplicate rows")
print("Sales     :", sales.duplicated().sum())
print("SKU       :", sku.duplicated().sum())
print("Inventory :", inventory.duplicated().sum())
print("Calendar  :", calendar.duplicated().sum())

print("\nDuplicate SKU IDs in master:", sku["sku_id"].duplicated().sum())
print("Duplicate calendar dates:", calendar["date"].duplicated().sum())


## 8. Remove exact duplicate rows

In [ ]:
sales = sales.drop_duplicates().reset_index(drop=True)
sku = sku.drop_duplicates().reset_index(drop=True)
inventory = inventory.drop_duplicates().reset_index(drop=True)
calendar = calendar.drop_duplicates().reset_index(drop=True)

print("Duplicates removed.")
print("Sales:", sales.shape)
print("SKU:", sku.shape)
print("Inventory:", inventory.shape)
print("Calendar:", calendar.shape)


## 9. Validate key columns

In [ ]:
required_columns = {
    "sales": ["date", "sku_id", "units_sold", "revenue", "unit_price", "promotion"],
    "sku": ["sku_id", "category", "subcategory", "unit_price", "cost_price", "brand"],
    "inventory": ["store_id", "sku_id", "stock_on_hand", "reorder_point", "safety_stock"],
    "calendar": ["date", "week", "month", "season", "holiday_flag", "promotion_events"]
}

datasets = {
    "sales": sales,
    "sku": sku,
    "inventory": inventory,
    "calendar": calendar
}

for name, cols in required_columns.items():
    missing_cols = [c for c in cols if c not in datasets[name].columns]
    if missing_cols:
        print(f"{name}: missing columns -> {missing_cols}")
    else:
        print(f"{name}: all required columns present")


## 10. Check invalid numeric values

In [ ]:
checks = {
    "sales_negative_units": (sales["units_sold"] < 0).sum(),
    "sales_negative_revenue": (sales["revenue"] < 0).sum(),
    "sales_nonpositive_price": (sales["unit_price"] <= 0).sum(),
    "inventory_negative_stock": (inventory["stock_on_hand"] < 0).sum(),
    "inventory_negative_reorder_point": (inventory["reorder_point"] < 0).sum(),
    "inventory_negative_safety_stock": (inventory["safety_stock"] < 0).sum(),
    "sku_negative_unit_price": (sku["unit_price"] < 0).sum(),
    "sku_negative_cost_price": (sku["cost_price"] < 0).sum(),
}

display(pd.Series(checks, name="count").to_frame())


## 11. Handle missing values

In [ ]:
# Preserve the raw values first; only apply safe, documented cleaning rules.

# Sales: rows without a date or SKU cannot be reliably used in analysis.
sales = sales.dropna(subset=["date", "sku_id"]).copy()

# Calendar: date is its primary key, so rows without date cannot be joined.
calendar = calendar.dropna(subset=["date"]).copy()

# SKU master: SKU ID is the key.
sku = sku.dropna(subset=["sku_id"]).copy()

# Inventory: SKU ID is required to connect inventory to products.
inventory = inventory.dropna(subset=["sku_id"]).copy()

print("Required-key missing rows removed.")


## 12. Validate SKU relationships

In [ ]:
sales_skus = set(sales["sku_id"].dropna().unique())
master_skus = set(sku["sku_id"].dropna().unique())
inventory_skus = set(inventory["sku_id"].dropna().unique())

print("Unique SKUs in sales:", len(sales_skus))
print("Unique SKUs in master:", len(master_skus))
print("Unique SKUs in inventory:", len(inventory_skus))

sales_not_in_master = sales_skus - master_skus
inventory_not_in_master = inventory_skus - master_skus

print("Sales SKUs not in master:", len(sales_not_in_master))
print("Inventory SKUs not in master:", len(inventory_not_in_master))

if sales_not_in_master:
    print("Example missing sales SKUs:", list(sales_not_in_master)[:10])
if inventory_not_in_master:
    print("Example missing inventory SKUs:", list(inventory_not_in_master)[:10])


## 13. Validate calendar relationship

In [ ]:
sales_dates = set(sales["date"].dropna().unique())
calendar_dates = set(calendar["date"].dropna().unique())

sales_dates_not_in_calendar = sales_dates - calendar_dates

print("Unique sales dates:", len(sales_dates))
print("Unique calendar dates:", len(calendar_dates))
print("Sales dates missing from calendar:", len(sales_dates_not_in_calendar))

if sales_dates_not_in_calendar:
    print("Example missing dates:", list(sales_dates_not_in_calendar)[:10])


## 14. Merge sales + SKU master + calendar

In [ ]:
# Many-to-one merge: each sales row should map to one SKU master row.
sales_sku = sales.merge(
    sku,
    on="sku_id",
    how="left",
    suffixes=("_sales", "_master"),
    validate="many_to_one"
)

# Many-to-one merge: each sales date should map to one calendar row.
sales_complete = sales_sku.merge(
    calendar,
    on="date",
    how="left",
    validate="many_to_one"
)

print("Sales + SKU shape:", sales_sku.shape)
print("Final sales analysis dataset shape:", sales_complete.shape)
display(sales_complete.head())


## 15. Check merge results

In [ ]:
merge_columns_to_check = [
    "sku_name", "category", "subcategory",
    "brand", "week", "month", "season",
    "holiday_flag", "promotion_events"
]

available = [c for c in merge_columns_to_check if c in sales_complete.columns]

display(
    sales_complete[available].isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_after_merge")
)


## 16. Create basic date features

In [ ]:
sales_complete["year"] = sales_complete["date"].dt.year
sales_complete["month_num"] = sales_complete["date"].dt.month
sales_complete["day"] = sales_complete["date"].dt.day
sales_complete["day_of_week"] = sales_complete["date"].dt.dayofweek
sales_complete["is_weekend"] = (sales_complete["day_of_week"] >= 5).astype(int)
sales_complete["week_start"] = (
    sales_complete["date"]
    - pd.to_timedelta(sales_complete["day_of_week"], unit="D")
)

display(
    sales_complete[
        ["date", "year", "month_num", "day", "day_of_week",
         "is_weekend", "week_start"]
    ].head(10)
)


## 17. Final data-quality summary

In [ ]:
summary = pd.DataFrame({
    "dataset": ["sales", "sku", "inventory", "calendar", "sales_complete"],
    "rows": [
        len(sales),
        len(sku),
        len(inventory),
        len(calendar),
        len(sales_complete)
    ],
    "columns": [
        sales.shape[1],
        sku.shape[1],
        inventory.shape[1],
        calendar.shape[1],
        sales_complete.shape[1]
    ]
})

display(summary)

print("Final missing values in sales_complete:")
display(
    sales_complete.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


## 18. Save cleaned datasets

In [ ]:
# Create output directory if it does not exist.
os.makedirs(DATA_DIR, exist_ok=True)

# Save cleaned/analysis-ready data.
sales.to_pickle(os.path.join(DATA_DIR, "sales_clean.pkl"))
sku.to_pickle(os.path.join(DATA_DIR, "sku_clean.pkl"))
inventory.to_pickle(os.path.join(DATA_DIR, "inventory_clean.pkl"))
calendar.to_pickle(os.path.join(DATA_DIR, "calendar_clean.pkl"))
sales_complete.to_pickle(os.path.join(DATA_DIR, "sales_complete.pkl"))

# Also save CSV for easy inspection.
sales_complete.to_csv(
    os.path.join(DATA_DIR, "sales_complete.csv"),
    index=False
)

print("Cleaned datasets saved successfully.")


## Data-cleaning output

The main analysis-ready file is:

`data/sales_complete.pkl`

It combines:
- daily sales
- SKU/product information
- calendar information

Inventory remains as a separate cleaned table because it is a periodic inventory snapshot and should be joined to forecasting/risk calculations at the appropriate date/SKU level rather than blindly duplicating inventory across every sales row.

**Next notebook:** `02_eda.ipynb`
